# Online Retail Analysis

## Dataset
This dataset contains transactional data for a UK-based online retail company, covering the period from December 2010 to December 2011.

The dataset includes information about:
- products
- quantities
- prices
- customers
- countries
- timestamps

Source: UCI Machine Learning Repository

## Objective
The goal of this analysis is to explore purchasing behavior, identify patterns, and extract business insights from the data.

In [61]:
from turtledemo.chaos import plot

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from IPython.core.pylabtools import figsize
from matplotlib.pyplot import xlabel

df_original = pd.read_excel('Online Retail.xlsx')

def sep():
    print('='*70)

## Data Inspection

In [62]:
print("Size of dataset:", df_original.shape)
sep()
print(df_original.head())
sep()
df_original.info()
sep()
print(df_original.describe())
sep()
print(df_original.isna().sum())

Size of dataset: (541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------

### Conclusion
`CustomerID` has 132,603 missing values and `Description` has 1454 missing values. Additionally, dataset has negative values of columns which are supposed to be positive, e.g. `UnitPrice` and `Quantity`.


## Data Cleaning

In [ ]:
df = df_original

df_with_inappropriate_values = df_original[(df['Quantity'] <= 0) | (df['UnitPrice'] <= 0)]

print("Number of inappropriate rows:", df_with_inappropriate_values.shape[0])
print("Number of rows where UnitPrice < 0: ", df_with_inappropriate_values[df_with_inappropriate_values['UnitPrice'] < 0].shape[0])
print("Number or rows where Quantity <= 0:", df_with_inappropriate_values[df_with_inappropriate_values['Quantity'] <= 0].shape[0])

df_price0 = df[df['UnitPrice'] == 0]
print("Number of rows where UnitPrice == 0:", df_price0.shape[0])
print("Number of rows where UnitPrice == 0 and UserID in NaN:", df_price0['CustomerID'].isna().sum())
print("Number of rows where Quantity == 0:", (df_with_inappropriate_values['Quantity'] == 0).sum())

print(df_with_inappropriate_values.head())

### Conclusion
The dataset contains:
- negative quantities, which likely represent product returns
- zero or negative prices, which may indicate data errors or special transactions

We carefully filter the dataset to focus on valid sales transactions.

This ensures that our analysis reflects actual purchasing behavior.

## Feature Engineering
### Canceled transactions
Description of the dataset specifies that if `InvoiceNO` starts with `c`, the transaction was canceled. It is noticeable that rows with such `InvoiceNo` have negative number of `Quantity`.
### Next steps
- Remove rows from dataset according to the last conclusion
- Create the new column that states whether transaction is valid `bool`: `Valid`.
- Check whether valid transactions have only positive `Quantity` and not valid have only negative `Quantity`.
- In case of success, update rows with not valid transaction to have positive `Quantity`.

In [ ]:
df = df.drop(df[df['UnitPrice'] < 0].index)
rm_customer_NaN_price_0 = df[(df['UnitPrice'] == 0)]['CustomerID'].isna()
rm_customer_NaN_price_0 = rm_customer_NaN_price_0[rm_customer_NaN_price_0]
df = df.drop(rm_customer_NaN_price_0.index)
print(f"Updated dataset df have {df.shape[0]} rows")

def check_validity_by_InvoiceNo(x):
    return not str(x).startswith('C')

df['Valid'] = df['InvoiceNo'].apply(check_validity_by_InvoiceNo)
print(f"Dataset have {df['Valid'].sum()} valid (not cancelled) transactions.")

check_valid = df[(df['Valid']) & (df['Quantity'] <= 0)]
check_not_valid = df[(df['Valid'] == False) & (df['Quantity'] >= 0)]
print("Number of valid rows with Quantity <= 0:", check_valid.shape[0])
print("Number of not valid rows with Quantity >= 0:", check_not_valid.shape[0])

df['Quantity'] = df['Quantity'].apply(lambda x: abs(x))

### Conclusion
Dataset had negative values of quantity only for not valid transactions and positive number of quantity only for valid transactions. Now dataset have only positive column `Quantity` and column `Valid` that states whether transaction was not canceled.
### Next steps
- Add new column: Revenue per purchase `int`: `Revenue` = `UnitPrice` * `Quantity`
- Reinspect data

In [ ]:
df['Revenue'] = df['UnitPrice'] * df['Quantity']

print("Size of dataset:", df.shape)
sep()
print(df.head())
sep()
df.info()
sep()
print(df.describe())
sep()
print(df.isna().sum())

## Conclusion
The dataset contains no missing values in most columns including `Description`, but `CustomerID` still has a large number of missing entries (~20%).

## Pattern 1. Time of Purchase

We analyze how transaction frequency varies throughout the day.

In [ ]:
df['order_time_hour'] = df['InvoiceDate'].dt.hour
orders_per_hour = df.groupby('order_time_hour')['order_time_hour'].count() / 1000

orders_per_hour_bar_plot = sns.barplot(data=orders_per_hour)
orders_per_hour_bar_plot.set(xlabel = 'Hour', ylabel = 'Sales (1\'000)')
plt.title("Orders by Hour Distribution")
plt.show()

orders_12_15_oclock = orders_per_hour.loc[12:15].sum() * 1000
print(f"Purchase at 12-15 o'clock: {orders_12_15_oclock}. It takes up {round(orders_12_15_oclock / 1000 / orders_per_hour.sum() * 100, 1)}% of the total.")

### Conclusion
Most purchases occur between 12:00 and 15:00, indicating peak user activity during early afternoon hours.

This suggests that marketing campaigns or promotions could be most effective during this time window.

## Pattern 2. Country Distribution

In [ ]:
orders_per_country = df.groupby('Country')['Country'].count().sort_values(ascending=False)
df_orders_per_country = pd.DataFrame({'Country': orders_per_country.index, 'Orders': orders_per_country.values})
fig_1, axes_1 = plt.subplots(1, 3, figsize = (12, 4))

green_colors = ['darkgreen'] + ['limegreen'] * 3 + ['palegreen'] * 12

sns.barplot(data=df_orders_per_country.iloc[0:12], ax=axes_1[0], x='Country', y='Orders', hue='Country', palette=green_colors[:12])
axes_1[0].set_xticklabels(axes_1[0].get_xticklabels(), rotation=45, ha='right')
sns.barplot(data=df_orders_per_country.iloc[1:13], ax=axes_1[1], x='Country', y='Orders', hue='Country', palette=green_colors[1:13])
axes_1[1].set_xticklabels(axes_1[1].get_xticklabels(), rotation=45, ha='right')
sns.barplot(data=df_orders_per_country.iloc[4:16], ax=axes_1[2], x='Country', y='Orders', hue='Country', palette=green_colors[4:16])
axes_1[2].set_xticklabels(axes_1[2].get_xticklabels(), rotation=45, ha='right')

GB_orders_percent_global = orders_per_country.loc['United Kingdom'] / orders_per_country.sum()
print(f"Percentage of orders from United Kingdom among all orders: {round(GB_orders_percent_global * 100, 2)}%")
Germany_France_EIRE_orders_percent_global = orders_per_country.iloc[1:4].sum() / orders_per_country.sum()
print(f"Percentage of orders from Germany, France and EIRE combined among all orders: {round(Germany_France_EIRE_orders_percent_global * 100, 2)}%")
Germany_France_EIRE_orders_percent_local = orders_per_country.iloc[1:4].sum() / orders_per_country.iloc[1:].sum()
print(f"Percentage of orders from Germany, France and EIRE combined among all orders except from United Kingdom: {round(Germany_France_EIRE_orders_percent_local * 100, 2)}%")

fig_1.subplots_adjust(wspace=0.3)

fig_1.suptitle("Orders by Country Distribution", fontsize=14)
axes_1[0].set_title("Top Countries by Orders")
axes_1[1].set_title("Excluding UK")
axes_1[2].set_title("Smaller Markets")

### Conclusion
The United Kingdom accounts for approximately 91% of all transactions, indicating a highly concentrated market.

Other countries such as Germany, France, and Ireland contribute a much smaller but promising share (56% of all not United Kingdom transactions).

This suggests:
- strong dominance in the UK market
- potential opportunity for international expansion

## Pattern 3. Cancellation Rates by Country

In [ ]:
df_orders_per_country = pd.DataFrame(orders_per_country)
df_orders_per_country.columns = ['Orders']
df_orders_per_country['Cancelled_orders'] = df_orders_per_country['Orders'] - df.groupby('Country')['Valid'].sum()
df_orders_per_country['Cancellation_rate'] =  df_orders_per_country['Cancelled_orders'] / df_orders_per_country['Orders']
country_cancellation_rate_top = df_orders_per_country['Cancellation_rate'].sort_values(ascending=False)
df_cancellation_per_country = pd.DataFrame({'Country': country_cancellation_rate_top.index, 'Cancellation Rate': country_cancellation_rate_top.values})

fig_2, axes_2 = plt.subplots(1, 3, figsize = (12, 4))
red_colors = ['maroon'] + ['red'] * 4 + ['salmon'] * 12

sns.barplot(data=df_cancellation_per_country.iloc[0:12], ax=axes_2[0], x='Country', y='Cancellation Rate', hue='Country', palette=red_colors[:12])
axes_2[0].set_xticklabels(axes_2[0].get_xticklabels(), rotation=45, ha='right')
sns.barplot(data=df_cancellation_per_country.iloc[1:13], ax=axes_2[1], x='Country', y='Cancellation Rate', hue='Country', palette=red_colors[1:13])
axes_2[1].set_xticklabels(axes_2[1].get_xticklabels(), rotation=45, ha='right')
sns.barplot(data=df_cancellation_per_country.iloc[5:17], ax=axes_2[2], x='Country', y='Cancellation Rate', hue='Country', palette=red_colors[5:17])
axes_2[2].set_xticklabels(axes_2[2].get_xticklabels(), rotation=45, ha='right')

highest_cancellation_rate = df_cancellation_per_country.iloc[0]['Cancellation Rate']
group_cancellation_rate_avg = df_cancellation_per_country.iloc[1:5]['Cancellation Rate'].mean()

print(f'The highest cancellation rate is {round(highest_cancellation_rate * 100, 2)}% and relates to USA.')
print(f'The group with the highest cancellation rate includes Czech Republic, Malta, Japan, Saudi Arabia, and has average cancellation rate: {round(group_cancellation_rate_avg * 100, 2)}%.')

fig_2.subplots_adjust(wspace=0.3)

fig_2.suptitle("Cancellation Rate by Country Distribution", fontsize=14)
axes_2[0].set_title("Top Countries by Cancellation Rate")
axes_2[1].set_title("Excluding USA")
axes_2[2].set_title("Other Countries")

### Conclusion
The USA has the highest cancellation rate (~38%), which may indicate:
- logistical issues
- customer dissatisfaction
- or specific market challenges

Czech Republic, Malta, Japan, Saudi Arabia also show elevated cancellation rates.


## Pattern 4. Customer Behavior

In [ ]:
df_orders_per_user = df.groupby('CustomerID').agg({
    'InvoiceNo': 'count',
    'Valid': 'sum'
})
df_orders_per_user.columns = ["Orders", "Valid orders"]
df_orders_per_user['Not valid orders'] = df_orders_per_user['Orders'] - df_orders_per_user['Valid orders']
df_orders_per_user['Cancellation rate'] = df_orders_per_user['Not valid orders'] / df_orders_per_user['Orders']
user_cancellation_rate = df_orders_per_user['Cancellation rate'].sort_values(ascending=False)

user_high_cancellation_rate = df_orders_per_user[(df_orders_per_user["Orders"] >= 10) & (df_orders_per_user['Cancellation rate'] >= 0.4)]
print(f"User with high cancellation rate ({user_high_cancellation_rate.shape[0]}):")
print(user_high_cancellation_rate.index)

### Conclusion
A group of 18 customers shows unusually high cancellation rates (≥40%) with at least 10 transactions.

This may indicate:
- fraudulent behavior
- bulk ordering followed by cancellations
- or abnormal usage patterns

These users should be further investigated.

## Pattern 5. Top products

In [ ]:
productDesc_revenue_top = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False)

blue_colors = ['dodgerblue'] * 1 + ['lightskyblue'] * 19

df_productDesc_revenue_top = pd.DataFrame({'Description': productDesc_revenue_top.index, 'Revenue': productDesc_revenue_top.values})

productDesc_revenue_top_plot = sns.barplot(data=df_productDesc_revenue_top.iloc[:20], x='Description', y='Revenue', hue='Description', palette=blue_colors)
productDesc_revenue_top_plot.set_xticklabels(productDesc_revenue_top_plot.get_xticklabels(), rotation=45, ha='right')

print(f"Product with the highest revenue: {df_productDesc_revenue_top.iloc[0]['Description']}; {df_productDesc_revenue_top.iloc[0]['Revenue']}£")

plt.title("Top Products by Total Revenue")

### Conclusion
Product "PAPER CRAFT , LITTLE BIRDIE" achieved the highest revenue: 336939.2£.

## Final Insights

- UK dominates sales (91%)
- Peak purchase time: 12–15
- PAPER CRAFT , LITTLE BIRDIE generate the highest revenue
- Some countries show high cancellation rates
- Suspicious user group identified